# M6 — Code TF-IDF + Svm linéaire

Ce modèle utilise TF-IDF pour représenter le code source des solutions sous forme de vecteurs, puis un SVM linéaire pour prédire les huit tags de chaque exercice. La vectorisation repose sur les tokens et les caractéristiques lexicales du code, sans utiliser les énoncés. Après l'entraînement, plusieurs configurations d'hyperparamètres sont comparées sur l'ensemble de validation afin de sélectionner la meilleure selon le F1 macro. Les seuils de décision sont ensuite ajustés pour chaque tag, puis le modèle est évalué et sauvegardé. L'ensemble de test reste réservé à l'évaluation finale.

## 1. Charger et contrôler les données

Seul le code source de la solution sert d’entrée. Les énoncés, identifiants et annotations ne sont jamais transmis au vectoriseur. 

In [1]:
import hashlib
import json
import platform
import warnings
from importlib.metadata import version
from pathlib import Path
from time import perf_counter

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    accuracy_score, f1_score, hamming_loss,
    precision_recall_fscore_support,
)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC
from threadpoolctl import threadpool_limits

racine = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "entrainement.jsonl").is_file()), None
)
if racine is None:
    raise FileNotFoundError("Ouvrez ce notebook depuis le dépôt du projet.")

In [2]:
dossier_modele = racine / "models" / "m6_code_tfidf_linear_svm"
dossier_modele.mkdir(parents=True, exist_ok=True)
graine = 42
tags_cibles = ["math", "graphs", "strings", "number theory",
               "trees", "geometry", "games", "probabilities"]
colonnes_cibles = ["cible_" + tag.replace(" ", "_") for tag in tags_cibles]
champ_code = "source_code"
versions = {nom: version(nom) for nom in
            ["numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "joblib", "threadpoolctl"]}
empreintes = {}

def charger_ensemble(nom):
    chemin = racine / "data" / f"{nom}.jsonl"
    contenu = chemin.read_bytes()
    empreintes[nom] = hashlib.sha256(contenu).hexdigest()
    return pd.DataFrame([json.loads(ligne) for ligne in contenu.decode("utf-8").splitlines()
                         if ligne.strip()])

entrainement = charger_ensemble("entrainement")
validation = charger_ensemble("validation")
for donnees in [entrainement, validation]:
    assert donnees[colonnes_cibles].isin([0, 1]).all().all(), "Cibles non binaires."
    assert champ_code in donnees.columns, "Le champ source_code est absent."
    assert donnees[champ_code].map(lambda code: isinstance(code, str) and bool(code.strip())).all(), \
        "Un code source est absent ou vide."
    assert not donnees[champ_code].duplicated().any(), "Des codes identiques sont présents."
assert set(entrainement[champ_code]).isdisjoint(validation[champ_code]), \
    "Un même code apparaît dans l’entraînement et la validation."
y_train = entrainement[colonnes_cibles].to_numpy(dtype=np.int64)
y_validation = validation[colonnes_cibles].to_numpy(dtype=np.int64)
for champ in ["fichier_source", "src_uid", "code_uid"]:
    assert not entrainement[champ].isna().any() and not validation[champ].isna().any()
    assert not entrainement[champ].duplicated().any()
    assert not validation[champ].duplicated().any()
    assert set(entrainement[champ]).isdisjoint(validation[champ]), f"Fuite : {champ}"
for cibles in [y_train, y_validation]:
    assert ((cibles.sum(axis=0) > 0) & (cibles.sum(axis=0) < len(cibles))).all()

warnings.filterwarnings("error", category=ConvergenceWarning)
print(f"{len(entrainement)} exercices d’entraînement et {len(validation)} de validation.")

copies_train = entrainement[champ_code].str.replace(r"\s+", "", regex=True)
copies_validation = validation[champ_code].str.replace(r"\s+", "", regex=True)
display(pd.DataFrame([
    {"controle": "Codes répétés hors espaces — entraînement",
     "nombre": int(copies_train.duplicated().sum())},
    {"controle": "Codes répétés hors espaces — validation",
     "nombre": int(copies_validation.duplicated().sum())},
    {"controle": "Codes communs hors espaces — entraînement / validation",
     "nombre": len(set(copies_train) & set(copies_validation))},
]))

3470 exercices d’entraînement et 739 de validation.


,controle,nombre
0,Codes répétés hors espaces — entraînement,0
1,Codes répétés hors espaces — validation,0
2,Codes communs hors espaces — entraînement / va...,0


## 2. Construire le TF-IDF du code

Les codes restent intacts : casse, commentaires et chaînes sont conservés, sans
décodage HTML. Une expression régulière reconnaît les identifiants et mots-clés,
les nombres, les opérateurs composés et les signes. Il s’agit d’une découpe lexicale simple, pas d’une analyse syntaxique. Les tokens présents dans les commentaires et les chaînes sont également représentés.

In [4]:
motif_tokens = (
    r"(?u)[^\W\d]\w*|\d+(?:\.\d+)?(?:[eE][+-]?\d+)?|"
    r"(?:\*\*=?|//=?|<<=?|>>=?|:=|==|!=|<=|>=|->|[+\-*/%&|^]=)|[^\s\w]"
)
codes_train = entrainement[champ_code]
codes_validation = validation[champ_code]
parametres_tfidf = {
    "ngram_range": (1, 2), "min_df": 2, "max_df": 0.98, "max_features": 80000,
    "sublinear_tf": True, "norm": "l2", "lowercase": False,
    "token_pattern": motif_tokens,
}
vectoriseur = TfidfVectorizer(**parametres_tfidf)
debut = perf_counter()
X_train = vectoriseur.fit_transform(codes_train)
X_validation = vectoriseur.transform(codes_validation)
duree_vectorisation = perf_counter() - debut
assert (X_train.getnnz(axis=1) > 0).all(), "Un code d’entraînement n’a aucun token retenu."
assert (X_validation.getnnz(axis=1) > 0).all(), "Un code de validation n’a aucun token connu."
assert X_train.shape[1] == X_validation.shape[1]
print(f"{X_train.shape[1]:,} variables TF-IDF.")
display(pd.Series(parametres_tfidf, name="Valeur").to_frame())

33,665 variables TF-IDF.


,Valeur
ngram_range,"(1, 2)"
min_df,2
max_df,0.98
max_features,80000
sublinear_tf,True
norm,l2
lowercase,False
token_pattern,(?u)[^\W\d]\w*|\d+(?:\.\d+)?(?:[eE][+-]?\d+)?|...


## 3. Comparer six configurations

Huit classifieurs binaires sont entraînés via `OneVsRestClassifier`.
Les six couples `C ∈ {0.1, 1, 10}` et `class_weight ∈ {None, balanced}`
sont comparés avec le seuil par défaut. Le F1 macro donne le même poids
aux huit tags, le F1 micro complète la lecture globale.

In [5]:
def evaluer_predictions(reels, predictions):
    return {
        "f1_macro": float(f1_score(reels, predictions, average="macro", zero_division=0)),
        "f1_micro": float(f1_score(reels, predictions, average="micro", zero_division=0)),
        "hamming_loss": float(hamming_loss(reels, predictions)),
        "exact_match_ratio": float(accuracy_score(reels, predictions)),
    }

In [6]:
seuil_defaut = 0.0

def creer_classifieur(C, ponderation):
    estimateur = LinearSVC(C=C, class_weight=ponderation, dual="auto", max_iter=10000, random_state=graine)
    return OneVsRestClassifier(estimateur, n_jobs=1)

def calculer_scores(classifieur, matrice):
    return classifieur.decision_function(matrice)

experiences = []
classifieurs = {}
with threadpool_limits(limits=1):
    for C in [0.1, 1.0, 10.0]:
        for nom_ponderation, ponderation in [("aucune", None), ("equilibree", "balanced")]:
            cle = f"C{C}_{nom_ponderation}"
            classifieur = creer_classifieur(C, ponderation)
            debut = perf_counter()
            classifieur.fit(X_train, y_train)
            duree = perf_counter() - debut
            scores = calculer_scores(classifieur, X_validation)
            predictions = (scores >= seuil_defaut).astype(int)
            classifieurs[cle] = classifieur
            experiences.append({"experience": cle, "C": C, "class_weight": nom_ponderation,
                                 "temps_entrainement_s": duree,
                                 **evaluer_predictions(y_validation, predictions)})

recherche = pd.DataFrame(experiences).sort_values(
    ["f1_macro", "C"], ascending=[False, True], kind="stable"
).reset_index(drop=True)
configuration_retenue = recherche.iloc[0]
classifieur_final = classifieurs[configuration_retenue["experience"]]
scores_validation = calculer_scores(classifieur_final, X_validation)
display(recherche.round(4))
print("Configuration retenue :", configuration_retenue["experience"])

,experience,C,class_weight,temps_entrainement_s,f1_macro,f1_micro,hamming_loss,exact_match_ratio
0,C0.1_equilibree,0.1,equilibree,0.3952,0.4222,0.5245,0.0920,0.4438
1,C1.0_equilibree,1.0,equilibree,0.5863,0.3904,0.4968,0.0809,0.4858
2,C10.0_equilibree,10.0,equilibree,1.6347,0.3255,0.4297,0.0831,0.4655
3,C10.0_aucune,10.0,aucune,0.9563,0.3170,0.4192,0.0820,0.4723
4,C1.0_aucune,1.0,aucune,0.3306,0.2923,0.4209,0.0731,0.5210
5,C0.1_aucune,0.1,aucune,0.1483,0.1152,0.2520,0.0773,0.5074


Configuration retenue : C0.1_equilibree


## 4. Ajuster les huit seuils

Le classifieur retenu est conservé sans réentraînement. Pour chaque tag, choisir
le seuil donnant le meilleur F1 parmi une grille de quantiles des scores de validation
et le seuil par défaut (0.0).

In [7]:
quantiles_seuils = [0.50, 0.70, 0.80, 0.85, 0.90, 0.925,
                   0.95, 0.965, 0.975, 0.985, 0.99, 0.995]
seuils = []
for indice in range(len(tags_cibles)):
    candidats = np.unique(np.append(
        np.quantile(scores_validation[:, indice], quantiles_seuils), seuil_defaut
    ))
    evaluations = [
        (f1_score(y_validation[:, indice], scores_validation[:, indice] >= seuil,
                  zero_division=0), float(seuil)) for seuil in candidats
    ]
    meilleur = min(evaluations, key=lambda r: (-r[0], abs(r[1] - seuil_defaut), r[1]))
    seuils.append(meilleur[1])
seuils = np.array(seuils)

## 5. Lire les résultats de validation

Comparer les seuils par défaut et ajustés, puis examiner chaque tag avec son support,
sa précision, son rappel et son F1. 

In [8]:
lignes_globales, lignes_tags = [], []
for nom, valeur in [("Aucun tag", 0), ("Tous les tags", 1)]:
    predictions = np.full_like(y_validation, valeur)
    lignes_globales.append({"regle": nom, **evaluer_predictions(
        y_validation, predictions
    )})
for regle, seuils_regle in [("Défaut", np.full(8, seuil_defaut)), ("Ajustés", seuils)]:
    predictions = (scores_validation >= seuils_regle).astype(int)
    lignes_globales.append({"regle": regle,
                           **evaluer_predictions(y_validation, predictions)})
    precisions, rappels, f1, supports = precision_recall_fscore_support(
        y_validation, predictions, average=None, zero_division=0
    )
    for i, tag in enumerate(tags_cibles):
        lignes_tags.append({
            "regle": regle, "tag": tag, "support": int(supports[i]),
            "seuil": float(seuils_regle[i]), "precision": precisions[i],
            "rappel": rappels[i], "f1": f1[i],
        })
bilan = pd.DataFrame(lignes_globales)
par_tag = pd.DataFrame(lignes_tags)
display(bilan.round(4))
display(par_tag.loc[par_tag["regle"].eq("Ajustés")].round(4).reset_index(drop=True))

sans_cible = y_validation.sum(axis=1) == 0
controle_sans_cible = []
for regle, seuils_regle in [("Défaut", np.full(8, seuil_defaut)), ("Ajustés", seuils)]:
    predictions = scores_validation[sans_cible] >= seuils_regle
    avec_prediction = predictions.any(axis=1)
    controle_sans_cible.append({
        "regle": regle,
        "exercices_sans_cible": int(sans_cible.sum()),
        "aucun_tag_predit": int((~avec_prediction).sum()),
        "au_moins_un_tag_predit": int(avec_prediction.sum()),
    })
print("Contrôle des exercices sans tag cible (décomptes) :")
display(pd.DataFrame(controle_sans_cible))

,regle,f1_macro,f1_micro,hamming_loss,exact_match_ratio
0,Aucun tag,0.0000,0.0000,0.0864,0.4587
1,Tous les tags,0.1500,0.1591,0.9136,0.0000
2,Défaut,0.4222,0.5245,0.0920,0.4438
3,Ajustés,0.4493,0.5219,0.0942,0.4384


,regle,tag,support,seuil,precision,rappel,f1
0,Ajustés,math,211,0.0000,0.5217,0.6256,0.5690
1,Ajustés,graphs,81,0.0000,0.5567,0.6667,0.6067
2,Ajustés,strings,63,0.0000,0.4382,0.6190,0.5132
3,Ajustés,number theory,52,0.0093,0.3784,0.5385,0.4444
4,Ajustés,trees,49,0.1622,0.5179,0.5918,0.5524
5,Ajustés,geometry,25,0.0000,0.4800,0.4800,0.4800
6,Ajustés,games,16,0.0935,0.5000,0.1250,0.2000
7,Ajustés,probabilities,14,-0.3735,0.1429,0.5714,0.2286


Contrôle des exercices sans tag cible (décomptes) :


,regle,exercices_sans_cible,aucun_tag_predit,au_moins_un_tag_predit
0,Défaut,339,194,145
1,Ajustés,339,190,149


## 6. Sauvegarder le modèle final et les résultats

Un unique artefact contient le vectoriseur, les huit classifieurs, les seuils,
l’ordre des tags, le champ d’entrée, les versions et les empreintes des données. Les trois CSV conservent la recherche et les métriques. Le modèle sauvegardé reste celui appris sur l’entraînement, les seuils ont été réglés sur la validation.

In [9]:
modele_final = {
    "modele": "M6", "vectoriseur": vectoriseur, "classifieur": classifieur_final,
    "seuils": seuils, "tags": tags_cibles, "champ_code": champ_code,
    "type_scores": "marges_de_decision",
    "pretraitement": {"code_inchange": True, "html_unescape": False},
    "temps_vectorisation_s": duree_vectorisation,
    "configuration": {"C": float(configuration_retenue["C"]),
                      "class_weight": None if configuration_retenue["class_weight"] == "aucune"
                      else "balanced"},
    "graine": graine, "versions": versions, "python": platform.python_version(),
    "empreintes_donnees": empreintes,
}
_ = joblib.dump(modele_final, dossier_modele / "modele.joblib", compress=3)
recherche.to_csv(dossier_modele / "recherche_hyperparametres.csv", index=False)
bilan.to_csv(dossier_modele / "metriques_validation.csv", index=False)
par_tag.to_csv(dossier_modele / "metriques_par_tag_validation.csv", index=False)